# Notebook 01: Experiment with managed MLflow and scale with SageMaker AI jobs

<div style="border: 2px solid #ff9900; border-radius: 8px; padding: 15px; background-color: #fff3e0; margin-bottom: 10px;">
<strong>⚠️ Compatibility Notice:</strong> Tested with <strong>SageMaker Distribution Image 4.3.2</strong> and <strong>SageMaker Python SDK 3.20.0</strong>.
</div>

In this notebook, you take an idea to a scaled, tracked, registered, and approved model in one pass, building on the environment from notebook 00. Serving that model is notebook 04.

|||
|---|---|
|00. Environment setup ||
|**01. Experiment in a notebook, then scale with SageMaker AI jobs** |**◄ YOU ARE HERE**|
|02. Automate model governance with EventBridge and Model Cards ||
|03. Operationalize with an ML pipeline and model registry ||
|04. Deploy to an endpoint and monitor ||
|05. Templatize staged deployment as a Project ||

## What you do here

1. **Part 1** — explore the bank-marketing dataset, engineer features, and train XGBoost locally in plain Python, tracking every run in [SageMaker managed MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html). No `sagemaker` job APIs and no registration: the prototype stays an experiment.
2. **Part 2** — call the reusable preprocess and train functions locally, then run preprocess, train, and explain as managed jobs with `@remote` on the custom XGBoost image (BYOC). Notebook 03 wraps the same functions with `@step`.
3. Write the processed train, validation, test, and monitoring-baseline datasets to S3 as the canonical inputs for notebooks 03 and 04.
4. Explain the managed model with SHAP and standardized fairness metrics.
5. Register one deployable Model Package via inference-spec logging and `mlflow.register_model()`.
6. Stage the version as the data scientist, then promote and approve it as the governance officer, with IAM denying the data scientist that step.

**Outcome:** one registered Model Package version, approved for production, which notebook 04 deploys from the registry.

<div style="border: 4px solid #d32f2f; border-radius: 10px; padding: 20px; background-color: #ffebee; margin: 15px 0;">

### ⛔ IMPORTANT: Python 3.12 Required

All notebooks in this workshop **must** run on **Python 3.12**. The cell below will fail with an `AssertionError` if you are using a different Python version.

If you see an error, follow these steps:

1. Open a **Terminal** in JupyterLab (**File → New → Terminal**)
2. Run the setup script:
   ```bash
   cd ~/aim-408-end-to-end-modelops-on-sagemaker-ai
   ./setup-py312.sh
   ```
3. **Refresh your browser** to pick up the new kernel
4. In this notebook, go to **Kernel → Change Kernel → Python 3.12**
5. **Restart the kernel** and re-run from the [00-start-here.ipynb](./00-start-here.ipynb) cell to re-install the dependencies.

</div>

In [ ]:
import sys

assert sys.version_info[:2] == (3, 12), f"Expected Python 3.12, got {sys.version_info.major}.{sys.version_info.minor}. Please make sure you run on Python 3.12"

### Check the installed SageMaker version

In [ ]:
import sagemaker
import sagemaker.core
import sagemaker.serve
import sagemaker.mlops
from importlib.metadata import version

print(f"sagemaker: {version('sagemaker')}")
print(f"sagemaker.core: {version('sagemaker.core')}")
print(f"sagemaker.serve: {version('sagemaker.serve')}")
print(f"sagemaker.mlops: {version('sagemaker.mlops')}")

### Import packages
Import the notebook dependencies and the reusable preprocess, train, and explain functions shared with notebook 03.

In [ ]:
import os
import time
import json
import tempfile
import warnings
from io import StringIO
from html import escape
from urllib.parse import quote
from importlib.metadata import version

import boto3
import numpy as np
import pandas as pd
import xgboost as xgb
import matplotlib.pyplot as plt
import mlflow
import sagemaker_mlflow
from mlflow import MlflowClient
from IPython.display import HTML, display
from sklearn import metrics
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import roc_auc_score

from sagemaker.core.resources import MlflowApp, ModelPackage
from sagemaker.core.s3 import S3Downloader
from sagemaker.core.s3.client import s3_path_join
from sagemaker.core.common_utils import name_from_base
from sagemaker.core.utils.utils import configure_logging
from sagemaker.core.helper.session_helper import Session
from sagemaker.core.remote_function.client import remote
from sagemaker.core.remote_function.custom_file_filter import CustomFileFilter

# Notebook 01 runs these functions with @remote. Notebook 03 wraps the exact
# same implementations with @step.
from pipeline_steps.preprocess import preprocess
from pipeline_steps.train_fn import train
from pipeline_steps.explain import explain
# Two-persona approval gate (data scientist vs governance officer), shared with notebook 03.
from governance.approval_gate import (
    DATA_SCIENTIST,
    GOVERNANCE_OFFICER,
    approve_for_deployment,
    assume_persona,
    deny_statements,
    describe_guardrail,
    freeze_model_package_group,
    load_persona_policy,
    model_package_group_tags,
    print_lifecycle,
    set_lifecycle,
    set_version_metadata,
    simulate_lifecycle_decision,
    tag_model_package_group,
    try_update_model_package,
    wait_for_lifecycle,
)

warnings.filterwarnings("ignore")
configure_logging("INFO")

Load the variables stored by previous notebooks.

In [ ]:
%store -r 

%store

try:
    initialized
except NameError:
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")
    print("[ERROR] YOU HAVE TO RUN 00-start-here notebook   ")
    print("+++++++++++++++++++++++++++++++++++++++++++++++++")

Set and store the name of the dataset's target column.

In [ ]:
target_col = "y"
%store target_col

## Set up experiment tracking with managed MLflow
[SageMaker managed MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html) runs MLflow as a serverless app. [MLflow tracking](https://mlflow.org/docs/latest/tracking.html) records the inputs, parameters, configurations, and models of each iteration as `experiments` and `runs` (see [MLflow concepts](https://mlflow.org/docs/latest/tracking.html#concepts)).

Notebook 00 created (or reused) the MLflow app and stored its ARN in `mlflow_arn`. The cell below waits until the app is ready, which can take 2-3 minutes on a first run.

In [ ]:
mlflow_app = MlflowApp.get(arn=mlflow_arn)

max_wait_time = 300  # 5 minutes
start_time = time.time()

while mlflow_app.status in ["Creating", "Updating"]:
    if time.time() - start_time > max_wait_time:
        raise TimeoutError(f"MLflow app '{mlflow_name}' still {mlflow_app.status} after 5 minutes")
    print(f"MLflow app '{mlflow_name}' is {mlflow_app.status}... waiting 30 seconds")
    time.sleep(30)
    mlflow_app.refresh()

if mlflow_app.status not in ["Created", "Updated"]:
    raise RuntimeError(f"MLflow app '{mlflow_name}' is not usable (status: {mlflow_app.status})")

print(f"MLflow app '{mlflow_name}' is ready (status: {mlflow_app.status})")

Point MLflow at the app and set the experiment and registered-model names.

These are **shared names**: one experiment and one registered model for the whole workshop, suffixed with a short date stamp (`-DDMMYY`). They are created here on first use and persisted with `%store`, so every later notebook restores and reuses them. `set_experiment()` returns the existing experiment, and each `register_model()` call adds a new **version** rather than a new model.

In [ ]:
mlflow.set_tracking_uri(mlflow_arn)
print(f"Using MLflow server: {mlflow_arn}")

date_suffix = time.strftime("%d%m%y", time.gmtime())
experiment_name = f"from-idea-to-prod-experiment-{date_suffix}"
registered_model_name = f"from-idea-to-prod-model-{date_suffix}"

%store experiment_name
%store registered_model_name

experiment = mlflow.set_experiment(experiment_name=experiment_name)

print(f"Experiment: {experiment_name}")
print(f"Registered model: {registered_model_name}")

---
# Part 1 — Prototype in the notebook

## Load data
The dataset is already available locally from notebook 00.

In [ ]:
file_name = "bank-additional-full.csv"
input_path = "./data/bank-additional" 
output_path = "./data"

## EDA
Explore the dataset before modeling.

In [ ]:
df_data = pd.read_csv(os.path.join(input_path, file_name), sep=";")

pd.set_option("display.max_columns", 500)  # View all of the columns
df_data  # show first 5 and last 5 rows of the dataframe

Check the column metadata for null counts and data types.

In [ ]:
df_data.info()

Column statistics are a quick sanity check: age ranges 17-98 with a mean of 40, all reasonable.

In [ ]:
df_data.describe()

The target distribution is unbalanced: far more "no" than "yes" responses to the marketing calls.

In [ ]:
df_data[target_col].value_counts().plot.bar()
plt.show()

Check for missing values. This dataset is clean.

In [ ]:
df_data.isna().sum()

Distributions for the categorical columns: most customers are highly educated and married, and some have an unknown job.

In [ ]:
cat_columns = ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']

fig, axs = plt.subplots(3, 3, sharex=False, sharey=False, figsize=(20, 15))

counter = 0
for cat_column in cat_columns:
    value_counts = df_data[cat_column].value_counts()
    
    trace_x = counter // 3
    trace_y = counter % 3
    x_pos = np.arange(0, len(value_counts))
    
    axs[trace_x, trace_y].bar(x_pos, value_counts.values, tick_label = value_counts.index)
    
    axs[trace_x, trace_y].set_title(cat_column)
    
    for tick in axs[trace_x, trace_y].get_xticklabels():
        tick.set_rotation(90)
    
    counter += 1

plt.show()

Distributions for the numerical columns: call `duration` is almost always short, and most customers were not contacted in a previous campaign (`pdays`).

In [ ]:
num_columns = ['duration', 'campaign', 'pdays', 'previous']

fig, axs = plt.subplots(2, 2, sharex=False, sharey=False, figsize=(20, 15))

counter = 0
for num_column in num_columns:
    
    trace_x = counter // 2
    trace_y = counter % 2
    
    axs[trace_x, trace_y].hist(df_data[num_column])
    
    axs[trace_x, trace_y].set_title(num_column)
    
    counter += 1

plt.show()

Compare a column against the target, here marital status.

In [ ]:
j_df = pd.DataFrame()

j_df['yes'] = df_data[df_data[target_col] == 'yes']['marital'].value_counts()
j_df['no'] = df_data[df_data[target_col] == 'no']['marital'].value_counts()

j_df.plot.bar(title = 'Marital status and deposit')

## Feature engineering in the notebook
Prototype the transformations here; Part 2 calls the reusable implementation in `pipeline_steps/preprocess.py` with `@remote`.

The preprocessing:
1. Adds `no_previous_contact`: `1` when `pdays` is `999`, else `0`
1. Adds a column for whether the customer is working, from the `job` column
1. Removes the economic features, which would need high-precision forecasts at inference time
1. Removes `duration`, which is not known before a call
1. One-hot encodes categorical variables
1. Moves the target column `y` to the front

In [ ]:
# Indicator variable to capture when pdays takes a value of 999
df_data["no_previous_contact"] = np.where(df_data["pdays"] == 999, 1, 0)

# Indicator for individuals not actively employed
df_data["not_working"] = np.where(
    np.isin(df_data["job"], ["student", "retired", "unemployed"]), 1, 0
)

# remove unnecessary data
df_model_data = df_data.drop(
    ["duration", "emp.var.rate", "cons.price.idx", "cons.conf.idx", "euribor3m", "nr.employed"],
    axis=1,
)


bins = [18, 30, 40, 50, 60, 70, 90]
labels = ['18-29', '30-39', '40-49', '50-59', '60-69', '70-plus']

df_model_data['age_range'] = pd.cut(df_model_data.age, bins, labels=labels, include_lowest=True)
df_model_data = pd.concat([df_model_data, pd.get_dummies(df_model_data['age_range'], prefix='age', dtype=int)], axis=1)
df_model_data.drop('age', axis=1, inplace=True)
df_model_data.drop('age_range', axis=1, inplace=True)

scaled_features = ['pdays', 'previous', 'campaign']
df_model_data[scaled_features] = MinMaxScaler().fit_transform(df_model_data[scaled_features])

df_model_data = pd.get_dummies(df_model_data, dtype=int)  # Convert categorical variables to sets of indicators

# Replace "y_no" and "y_yes" with a single label column, and bring it to the front:
df_model_data = pd.concat(
    [
        df_model_data["y_yes"].rename(target_col),
        df_model_data.drop(["y_no", "y_yes"], axis=1),
    ],
    axis=1,
)

After preprocessing, re-inspect the dataset and column statistics as a sanity check.

In [ ]:
df_model_data

In [ ]:
df_model_data.describe()

## Split data

[SageMaker's XGBoost algorithm](https://docs.aws.amazon.com/sagemaker/latest/dg/xgboost.html#InputOutput-XGBoost), used in Part 2, expects libSVM or CSV with the target in the first column and no header row. The cells below shuffle and split the data into train, validation, and test sets, then save each as a correctly formatted CSV.

In [ ]:
df_shuffled = df_model_data.sample(frac=1, random_state=1729)
# Split data into 70% training, 20% validation, 10% test
train_idx = int(0.7 * len(df_shuffled))
val_idx = int(0.9 * len(df_shuffled))

train_data = df_shuffled.iloc[:train_idx]
validation_data = df_shuffled.iloc[train_idx:val_idx]
test_data = df_shuffled.iloc[val_idx:]

print(f"Data split > train:{train_data.shape} | validation:{validation_data.shape} | test:{test_data.shape}")

In [ ]:
train_data.to_csv(os.path.join(output_path, "train.csv"), index=False, header=False)
validation_data.to_csv(os.path.join(output_path, "validation.csv"), index=False, header=False)
test_data.to_csv(os.path.join(output_path, "test.csv"), index=False, header=False)

### Track the feature engineering in an MLflow run
Log data processing as well as training, so both live in the same experiment. Create a run with [`mlflow.start_run()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.start_run), then record into it with [`log_params()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_params) and [`log_artifact()`](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.log_artifact).

Here you log the split ratios and the **raw input dataset**. The train/validation/test splits are logged later by the Part 2 processing job, so the same CSVs are not uploaded twice.

In [ ]:
with mlflow.start_run(
    run_name=name_from_base("feature-engineering", short=True),
    description="feature engineering prototyped in the notebook (Part 1)"
) as run:
    mlflow.set_tags({"mlflow.user": user_profile_name})
    mlflow.log_params(
        {
            "train": 0.7,
            "validate": 0.2,
            "test": 0.1
        }
    )
    # Log the raw input dataset. The train/validation/test splits produced from it
    # are logged by the SageMaker processing job in Part 2 (same data, logged once).
    mlflow.log_artifact(local_path=os.path.join(input_path, file_name))

## Model training and validation

Part 1 uses the [open-source XGBoost](https://xgboost.ai/about) to train inside the notebook. First, split the features from the label in the training dataset.

In [ ]:
train_features = train_data.drop(target_col, axis=1)
train_label = pd.DataFrame(train_data[target_col])

Create the data structure XGBoost accepts as input.

In [ ]:
dtrain = xgb.DMatrix(train_features, label=train_label)

Set your hyperparameters and other variables.

In [ ]:
hyperparams = {
                "max_depth": 5,
                "eta": 0.5,
                "alpha": 2.5,
                "objective": "binary:logistic",
                "subsample" : 0.8,
                "colsample_bytree" : 0.8,
                "min_child_weight" : 3
              }

num_boost_round = 150
nfold = 3
early_stopping_rounds = 10

Run cross-validation on `nfold` folds to see how well the model generalizes with the current hyperparameters.

In [ ]:
cv_results = xgb.cv(
    params=hyperparams,
    dtrain=dtrain,
    num_boost_round=num_boost_round,
    nfold=nfold,
    early_stopping_rounds=early_stopping_rounds,
    metrics=["auc"],
    seed=10,
)

Fetch and print the metrics to understand how well the model performs.

In [ ]:
metrics_data = {
    "binary_classification_metrics": {
        "validation:auc": {
            "value": cv_results.iloc[-1]["test-auc-mean"],
            "standard_deviation": cv_results.iloc[-1]["test-auc-std"]
        },
        "train:auc": {
            "value": cv_results.iloc[-1]["train-auc-mean"],
            "standard_deviation": cv_results.iloc[-1]["train-auc-std"]
        },
    }
}

print(f"Cross-validated train-auc:{cv_results.iloc[-1]['train-auc-mean']:.2f}")
print(f"Cross-validated validation-auc:{cv_results.iloc[-1]['test-auc-mean']:.2f}")
display(metrics_data)

cv_results

Now retrain on the full training dataset rather than across folds.

### Train a model
Use [MLflow model flavors](https://mlflow.org/docs/latest/python_api/index.html#python-api) and [logging functions](https://mlflow.org/docs/latest/tracking/tracking-api.html#tracking-logging-functions) to log parameters, the model, metrics, and metadata to your experiment runs.

The test dataset drives early stopping, so first split its features and label and build the data structure the algorithm expects.

In [ ]:
test_features = test_data.drop(target_col, axis=1)
test_label = pd.DataFrame(test_data[target_col])
dtest = xgb.DMatrix(test_features, label=test_label)

The training loop below trains the model at several `max_depth` values and logs each run to MLflow for comparison.

In [ ]:
max_metric = 0.0
best_model_run_id = 0

with mlflow.start_run(
    run_name=name_from_base("training-", short=True),
    description=f"Fit estimator with different max_depth"
) as parent_run:
    mlflow.set_tags({'mlflow.user':user_profile_name})
    
    # Train the model for different max_depth values
    for index, depth in enumerate([2, 5, 10, 15, 20]):
        hyperparams["max_depth"] = depth
        print(f"Fit estimator with max_depth={depth}")
    
        with mlflow.start_run(
            run_name=f"max_depth={depth}",
            description=f"Fit estimator with max_depth={depth}",
            nested=True
        ) as child_run:
            mlflow.set_tags({'mlflow.user':user_profile_name})
            
            mlflow.xgboost.autolog(silent=True)
            
            # Train the model
            model = xgb.train(
                params=hyperparams, 
                dtrain=dtrain, 
                evals = [(dtrain,'train'), (dtest,'eval')], 
                num_boost_round=num_boost_round, 
                early_stopping_rounds=early_stopping_rounds, 
                verbose_eval = 0
            )
    
            # Calculate metrics
            test_auc = roc_auc_score(test_label, model.predict(dtest))
            train_auc = roc_auc_score(train_label, model.predict(dtrain))
            
            # Log hyperparameters and metrics to the run
            mlflow.log_params(hyperparams)
            mlflow.log_metrics({"test_auc":test_auc, "train_auc":train_auc}, step=index)
    
            if test_auc > max_metric:
                best_model_run_id = child_run.info.run_id
                max_metric = test_auc
    
            print(f"Test AUC: {test_auc:.4f} | Train AUC: {train_auc:.4f}")

### The prototype is not registered, and that is deliberate

MLflow can act as a [model registry](https://mlflow.org/docs/latest/model-registry.html), and because the app runs with `AutoModelRegistrationEnabled`, every `mlflow.register_model()` call also creates a Model Package in the [SageMaker Model Registry](https://docs.aws.amazon.com/sagemaker/latest/dg/model-registry.html). Registration is therefore a **governance action**: whatever lands in the registry is a candidate for approval and deployment.

A kernel-trained model does not qualify: not reproducible, no training-job lineage, no inference specification. Part 1 stops at tracked experiments; Part 2 registers exactly one model produced by the reusable `train()` function in a managed `@remote` job, with an inference specification attached.

For now, note which local run won the sweep.

In [ ]:
print(f"Best local run: {best_model_run_id} (test AUC {max_metric:.4f})")
print("Tracked, not registered — Part 2 registers the model produced by the managed @remote training job.")

### Open the best local model and compare the runs in MLflow

The training loop autologged one model for each child run. Use the first link to authenticate, the second to open the logged model selected by `best_model_run_id`, and the third to compare all local runs in **Chart view**. This is still a logged prototype model—not a registered model.

In [ ]:
sm_client = boto3.Session().client("sagemaker")
mlflow_client = MlflowClient()
experiment_id = mlflow.get_experiment_by_name(experiment_name).experiment_id

best_local_run = mlflow_client.get_run(best_model_run_id)
best_local_model_outputs = (
    best_local_run.outputs.model_outputs if best_local_run.outputs else []
)
if not best_local_model_outputs:
    raise RuntimeError(
        f"Best local run {best_model_run_id} has no autologged model output."
    )
best_local_model_id = best_local_model_outputs[0].model_id

presigned_url = sm_client.create_presigned_mlflow_app_url(
    Arn=mlflow_arn,
    ExpiresInSeconds=300,
    SessionExpirationDurationInSeconds=1800,
)["AuthorizedUrl"]
mlflow_ui_base = presigned_url.split("/auth")[0]
best_local_model_link = (
    f"{mlflow_ui_base}/#/experiments/{quote(str(experiment_id), safe='')}/models/"
    f"{quote(best_local_model_id, safe='')}/artifacts?workspace=default"
)
compare_run_link = (
    f"{mlflow_ui_base}/#/experiments/{quote(str(experiment_id), safe='')}/runs"
    "?workspace=default&compareRunsMode=CHART"
)
print(f"Best local logged model: {best_local_model_id} from run {best_model_run_id}")

In [ ]:
display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(presigned_url, quote=True)}">'
    '1. Open the MLflow UI and authenticate</a></b>'
))

In [ ]:
display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(best_local_model_link, quote=True)}">'
    '2. Open the best local logged model in MLflow</a></b>'
))

display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(compare_run_link, quote=True)}">'
    '3. Compare the local training runs in Chart view</a></b>'
))

## Explore experiments with the MLflow UI
Open the app from the Studio UI: **MLflow** widget > **Open MLflow**.

![](img/mlflow-apps-studio.png)

The UI opens in a new browser window. Select your workspace (*default*) and the experiment generated above to view its runs.

![](img/experiment-mlflow.png)

**Chart view** compares runs; the eye icons show or hide individual runs.

![](img/comparing-runs.png)

The **Models** tab lists registered models and their versions.

![](img/logged-models.png)

---
# Part 2 — Validate locally, then run on managed compute

Everything above was exploratory notebook code. Now switch to the reusable implementations in `pipeline_steps/`: invoke `preprocess()` and `train()` locally to validate their data/model contract, then apply `@remote` to the same functions. Only the remote training result continues to explainability and registration. Notebook 03 wraps these functions with `@step`.

## Run the same functions locally and remotely

The functions in `pipeline_steps/` are ordinary Python. Call them directly first, then use the [`@remote` decorator](https://docs.aws.amazon.com/sagemaker/latest/dg/train-remote-decorator.html) to serialize the function and its arguments, run it on SageMaker managed compute, and return its result to this notebook.

`@remote` always creates a SageMaker **TrainingJob**, including when the function performs data preparation. This notebook therefore calls the first managed operation *remote preprocessing*, not a SageMaker ProcessingJob. Use [`FrameworkProcessor`](https://docs.aws.amazon.com/sagemaker/latest/dg/processing-job-frameworks.html) when native `ProcessingInput` and `ProcessingOutput` semantics are required; notebook 04 uses that pattern for scheduled monitoring.

For script-first training with explicit channels and model-output artifacts, [`ModelTrainer`](https://sagemaker.readthedocs.io/en/stable/api/training/model_trainer.html) is the corresponding alternative. It is not executed here because it would add a second implementation and another TrainingJob for the same model.

In [ ]:
remote_instance_type = "ml.r5.xlarge"
managed_output_s3_prefix = s3_path_join("s3://", bucket_name, bucket_prefix)

# MLflow reads the first two variables directly. The other values are recorded
# by pipeline_steps/runtime_utils.py for runtime lineage.
remote_environment = {
    "MLFLOW_TRACKING_URI": mlflow_arn,
    "MLFLOW_EXPERIMENT_NAME": experiment_name,
    "CONTAINER_IMAGE": xgboost_image,
    "USER": user_profile_name,
    "REGION": region,
}

remote_file_filter = CustomFileFilter(
    ignore_name_patterns=[
        "data", "tmp", "img", "*.ipynb", "__pycache__", ".git", ".build"
    ]
)

remote_settings = {
    "instance_type": remote_instance_type,
    "role": sm_role,
    "image_uri": xgboost_image,
    "include_local_workdir": True,
    "custom_file_filter": remote_file_filter,
    "environment_variables": remote_environment,
    "keep_alive_period_in_seconds": 3600,
}

### Inspect `preprocess()`

Expand the panel to inspect the exact preprocessing function. The next cells call this function directly, then wrap the same function object with `@remote`. Its implementation remains in [`pipeline_steps/preprocess.py`](pipeline_steps/preprocess.py).

In [ ]:
import inspect
from html import escape
from IPython.display import HTML, display


def source_details(function):
    """Return a collapsed, HTML-escaped view of a live function's source."""
    qualified_name = f"{function.__module__}.{function.__name__}"
    signature = escape(str(inspect.signature(function)))
    source = escape(inspect.getsource(function))
    return HTML(
        f"""
        <details style="margin: 0.5rem 0 1rem; border: 1px solid var(--jp-border-color2, #ddd); border-radius: 6px; padding: 0.6rem 0.8rem;">
          <summary style="cursor: pointer; font-weight: 600;">
            View source: <code>{escape(qualified_name)}{signature}</code>
          </summary>
          <pre style="margin-top: 0.8rem; max-height: 32rem; overflow: auto; white-space: pre;"><code>{source}</code></pre>
        </details>
        """
    )


display(source_details(preprocess))

### Validate `preprocess()` locally

Run `preprocess()` as an ordinary Python function before submitting it to SageMaker. It reads the local raw CSV and writes its train, validation, test, and baseline outputs under `tmp/local-function-validation/` in a separate MLflow parent run.

In [ ]:
local_output_prefix = os.path.abspath("tmp/local-function-validation")
for split in ["train", "validation", "test", "baseline"]:
    os.makedirs(os.path.join(local_output_prefix, split), exist_ok=True)

# The functions read the same MLflow variables locally and in the remote jobs.
# Mark this invocation as local in its runtime tags.
os.environ.update({**remote_environment, "CONTAINER_IMAGE": "local"})

local_preprocess_result = preprocess(
    input_data_s3_path=os.path.join(input_path, file_name),
    output_s3_prefix=local_output_prefix,
    pipeline_run_name=f"notebook-01-local-validation-{date_suffix}",
)

print(f"Local train data: {local_preprocess_result['train_data']}")
print(f"Local features:   {len(local_preprocess_result['feature_names'])}")

### Run `preprocess()` on managed compute

Apply `@remote` to the same function. The managed call reads the raw CSV from S3, writes the processed files back to S3, and returns the same output contract as the local call. These S3 paths become the canonical inputs for training and notebooks 03–04.

In [ ]:
preprocess_settings = {
    **remote_settings,
    "job_name_prefix": "from-idea-to-prod-preprocess",
}
remote_preprocess = remote(**preprocess_settings)(preprocess)

preprocess_result = remote_preprocess(
    input_data_s3_path=s3_data_uri,
    output_s3_prefix=managed_output_s3_prefix,
    pipeline_run_name=f"notebook-01-managed-{date_suffix}",
)

In [ ]:
train_data_s3_path = preprocess_result["train_data"]
validation_data_s3_path = preprocess_result["validation_data"]
test_x_data_s3_path = preprocess_result["test_x_data"]
test_y_data_s3_path = preprocess_result["test_y_data"]
baseline_data_s3_path = preprocess_result["baseline_data"]

# Preserve the directory variables consumed later in this notebook and by
# notebooks 03 and 04.
train_s3_url = train_data_s3_path.rsplit("/", 1)[0]
validation_s3_url = validation_data_s3_path.rsplit("/", 1)[0]
test_s3_url = test_x_data_s3_path.rsplit("/", 1)[0]
baseline_s3_url = baseline_data_s3_path.rsplit("/", 1)[0]
feature_names = preprocess_result["feature_names"]

%store train_s3_url
%store validation_s3_url
%store test_s3_url
%store baseline_s3_url
# Read one output back as a quick contract check.
train_data = pd.read_csv(
    StringIO(S3Downloader.read_file(train_data_s3_path)),
    header=None,
)
print(f"Training data: {train_data.shape} from {train_data_s3_path}")
train_data.head()

### Inspect `train()`

Expand the panel to inspect the exact training function in [`pipeline_steps/train_fn.py`](pipeline_steps/train_fn.py). Both the local and managed calls use this function and the same hyperparameter dictionary.

In [ ]:
display(source_details(train))

### Validate `train()` locally

Run `train()` against the local preprocessing outputs. The resulting model and metrics are tracked in the local-validation MLflow parent run, but this model is never selected for explainability or registration.

In [ ]:
hyperparams = {
    "num_round": 50,
    "max_depth": 3,
    "eta": 0.5,
    "alpha": 2.5,
    "gamma": 0,
    "min_child_weight": 3,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "early_stopping_rounds": 10,
    "verbosity": 1,
}

In [ ]:
local_training_result = train(
    train_data_s3_path=local_preprocess_result["train_data"],
    validation_data_s3_path=local_preprocess_result["validation_data"],
    pipeline_run_id=local_preprocess_result["pipeline_run_id"],
    **hyperparams,
)

print(f"Local model ID:      {local_training_result['mlflow_model_id']}")
print(f"Local validation AUC: {local_training_result['validation_auc']:.4f}")
print("This local model is tracked for validation only; registration uses training_result below.")

### Run `train()` on managed compute

Apply `@remote` to the same function with the same hyperparameters. This call uses the managed preprocessing outputs and returns the MLflow model and run identifiers used by every downstream step.

In [ ]:
train_settings = {
    **remote_settings,
    "job_name_prefix": "from-idea-to-prod-train",
}
remote_train = remote(**train_settings)(train)

training_result = remote_train(
    train_data_s3_path=train_data_s3_path,
    validation_data_s3_path=validation_data_s3_path,
    pipeline_run_id=preprocess_result["pipeline_run_id"],
    **hyperparams,
)

### Select the managed model

Resolve the model by the ID returned from the managed `train()` call. The earlier `local_training_result` remains visible in MLflow as validation evidence, but it cannot feed this assignment or the registration path.

In [ ]:
mlflow_client = MlflowClient()
logged_model = mlflow_client.get_logged_model(training_result["mlflow_model_id"])
best_run_id = training_result["mlflow_run_id"]

assert logged_model.source_run_id == best_run_id, (
    f"Logged model source run {logged_model.source_run_id} does not match "
    f"the training result {best_run_id}"
)


def metric_of(model, key):
    logged_metrics = {m.key: m.value for m in (model.metrics or [])}
    return logged_metrics.get(key, training_result.get(key))


print(f"Managed model ID: {logged_model.model_id}")
print(f"Source run:       {best_run_id}")
print(f"Artifact URI:     {logged_model.artifact_location}")
print(f"Validation AUC:   {metric_of(logged_model, 'validation_auc'):.4f}")

## Explain the managed model before registration

Approval needs more than one AUC number. Before registration, produce a **feature-attribution and fairness report** for exactly the managed model you are about to register. Logging it onto that model's MLflow run means the Model Package inherits lineage that already includes the explanation.

The report uses **SHAP** for feature attribution and standardized fairness metrics computed with pandas. `shap.TreeExplainer` applies directly to the XGBoost booster and explains the complete test set without sampling.

### Inspect `explain()`

Expand the panel to inspect the `explain()` entry point. Plotting and fairness helpers remain in the full [`pipeline_steps/explain.py`](pipeline_steps/explain.py) module.

In [ ]:
display(source_details(explain))

<div class="alert alert-info">
<b>Why there is no local explainability run here.</b> The local model is validation-only and will not enter the registry. Running the report against it would create a second set of governance evidence for a different model. This notebook therefore inspects the function locally but executes it only for the managed model selected above. Notebook 03 separately invokes the same function locally while validating the pipeline functions.
</div>

### Prepare the managed-model inputs

Download the held-out test set produced by managed preprocessing and verify that its returned feature names match the data. Notebook 04 uses the same files to test the endpoint.

In [ ]:
!aws s3 cp $test_s3_url/test_x.csv tmp/test_x.csv
!aws s3 cp $test_s3_url/test_y.csv tmp/test_y.csv

Load the test files into memory for the contract checks below. The explainability job reads the same S3 objects directly.

In [ ]:
test_x = np.loadtxt("tmp/test_x.csv", delimiter=",")
test_y = np.loadtxt("tmp/test_y.csv", delimiter=",")

print(f"test_x: {test_x.shape} | test_y: {test_y.shape}")

### Feature names and the fairness facet

The remote preprocessing function writes header-less CSV files and returns the feature names separately. Verify that they match the columns produced in Part 1.

Fairness analysis needs a **facet**: a group whose treatment is compared with everyone else. Part 1's age binning left one-hot columns per age band, so `age_60-69` and `age_70-plus` together express *customer aged 60 or over*.

In [ ]:
part1_feature_names = [c for c in df_model_data.columns if c != target_col]
feature_names = preprocess_result["feature_names"]
facet_columns = ["age_60-69", "age_70-plus"]

assert feature_names == part1_feature_names, "Remote preprocessing changed the feature order"
assert len(feature_names) == test_x.shape[1], (
    f"{len(feature_names)} feature names vs {test_x.shape[1]} columns in test_x.csv"
)
missing_facets = [c for c in facet_columns if c not in feature_names]
assert not missing_facets, f"facet columns not in the feature set: {missing_facets}"

print(f"{len(feature_names)} features, e.g. {feature_names[:6]}")
print(f"Fairness facet: {facet_columns}")

%store feature_names

In [ ]:
%%writefile ./explainability/requirements.txt
# The BYOC image already ships xgboost, pandas, numpy, mlflow and s3fs.
# The explainability job only needs these two extra libraries.
shap
matplotlib

### Run `explain()` on managed compute

Apply `@remote` to `explain()` with the shared runtime settings. The only additional dependencies are `shap` and `matplotlib`, installed from [`explainability/requirements.txt`](explainability/requirements.txt). The report is logged to the managed model's run and nested under the parent run returned by managed preprocessing.

In [ ]:
explain_settings = {
    **remote_settings,
    "dependencies": "./explainability/requirements.txt",
    "job_name_prefix": "from-idea-to-prod-explain",
}
remote_explain = remote(**explain_settings)(explain)

explain_result = remote_explain(
    mlflow_run_id=best_run_id,
    test_x_data_s3_path=test_x_data_s3_path,
    test_y_data_s3_path=test_y_data_s3_path,
    feature_names=feature_names,
    output_s3_prefix=managed_output_s3_prefix,
    facet_columns=facet_columns,
    pipeline_run_id=preprocess_result["pipeline_run_id"],
)

### Read the report

The job logged the plots, the per-instance SHAP table, and the bias metrics to the model's MLflow run, and returned the headline numbers. Open the run in MLflow for the beeswarm, the global importance bar chart, and the waterfall for the most confident prediction.

Read `DI_disparate_impact` as the ratio of positive-prediction rates between the facet and the comparison group: `1.0` is parity, and the further from `1.0`, the more differently the model treats the two groups. `DPL` versus `DPPL` tells you whether a gap was already in the labels (pre-training) or introduced by the model (post-training).

Check `n_facet_group` before trusting either ratio: the age-60+ facet is only a few dozen of the ~4,100 test rows, so its rates move a lot with a handful of predictions. A large `DI` on a small facet is a prompt to collect more data for that group, not evidence of a broken model.

In [ ]:
print(f"Explained model:  {explain_result['explained_model_id']}")
print(f"E[y] (log-odds):  {explain_result['shap_expected_value_logodds']:.4f}")
print(f"Report in S3:     {explain_result['report_s3_prefix']}")

print("\nTop features by mean |SHAP|:")
for rank, feature in enumerate(explain_result["top_features"][:10], 1):
    print(f"  {rank:<3}{feature}")

print("\nFairness metrics for the age-60+ facet:")
for name, value in explain_result["bias_metrics"].items():
    print(f"  {name:<40}{value:.4f}" if isinstance(value, float) else f"  {name:<40}{value}")

## Explore the explainability results

In [ ]:
explain_run_link = (
    f"{presigned_url.split('/auth')[0]}"
    f"/#/experiments/{experiment_id}/runs/{explain_result['explainability_run_id']}?workspace=default"
)
display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(explain_run_link, quote=True)}">'
    'Open the explainability run in MLflow</a></b>'
))

## Explore the explainability artifacts

In [ ]:
explain_run_link = (
    f"{presigned_url.split('/auth')[0]}"
    f"/#/experiments/{experiment_id}/runs/{explain_result['explainability_run_id']}/artifacts?workspace=default"
)
display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(explain_run_link, quote=True)}">'
    'Open the explainability artifacts in MLflow</a></b>'
))

### Make the model deployable, then register it

A registered model is only useful if it can be *served*. The **inference-specification logging** pattern makes the logged model deployable:

1. Upload an `inference.py` into the logged model's artifact store under `code/`, with `MlflowClient.log_model_artifacts()`.
2. Log an **inference specification** on the logged model with the `sagemaker-mlflow` plugin (`>= 0.5.0`): the serving image, an `S3Prefix` `ModelDataSource` pointing at the artifact location, and the `SAGEMAKER_PROGRAM` / `SAGEMAKER_SUBMIT_DIRECTORY` environment variables.
3. Call `mlflow.register_model()`.

![MLflow logs the serving contract before registration creates a deployable SageMaker Model Package](img/inference-spec-registration-aws.svg)

The artifact store stays the single source of truth for the model bytes: no repacking, no copies, no `update_model_package` to patch the spec afterward.

<div class="alert alert-warning">
<b>Order matters:</b> log the specification <b>before</b> <code>mlflow.register_model()</code>. The auto-sync copies the spec onto the Model Package at registration time; log it afterward and the package is already created without it.
</div>

The reusable [`pipeline_steps/register.py`](pipeline_steps/register.py) step does exactly this in notebook 03. Here you do it inline, to see each piece.

#### Log an `inference.py` into the model's artifact store

The BYOC image serves with its own FastAPI stack (see [`byoc/serving_app.py`](byoc/serving_app.py)), which loads the script named by `SAGEMAKER_PROGRAM` and dispatches to `model_fn` / `input_fn` / `predict_fn` / `output_fn`. It has no default model loader, so a `model_fn` is mandatory: without it the container fails every `/ping` health check and the endpoint never reaches `InService`.

The `S3Prefix` `ModelDataSource` downloads everything under the artifact location to `/opt/ml/model`, so `code/inference.py` arrives at `/opt/ml/model/code/inference.py` and the MLflow model files (`MLmodel`, the booster, `conda.yaml`) land at `/opt/ml/model`, which is why `model_fn` can call `mlflow.xgboost.load_model(model_dir)`.

In [ ]:
INFERENCE_SCRIPT = r"""
import json

import numpy as np


def model_fn(model_dir):
    # The S3Prefix ModelDataSource downloaded the whole MLflow model directory
    # here, so MLflow can load the booster and resolve its serialization format.
    import mlflow.xgboost

    return mlflow.xgboost.load_model(model_dir)


def input_fn(request_body, request_content_type):
    if isinstance(request_body, (bytes, bytearray)):
        request_body = request_body.decode("utf-8")
    content_type = (request_content_type or "").split(";")[0].strip()

    if content_type == "text/csv":
        # One record per line, comma-separated features, no header and no label
        # column — the format preprocessing.py wrote for test_x.csv.
        rows = [r for r in request_body.strip().split("\n") if r]
        return np.array([[float(v) for v in r.split(",")] for r in rows])

    if content_type == "application/json":
        payload = json.loads(request_body)
        if isinstance(payload, dict):
            payload = payload.get("instances", payload.get("inputs", payload))
        array = np.array(payload, dtype=float)
        return array.reshape(1, -1) if array.ndim == 1 else array

    raise ValueError(f"Unsupported content type: {content_type}")


def predict_fn(input_data, model):
    import xgboost as xgb

    return model.predict(xgb.DMatrix(input_data))


def output_fn(prediction, accept):
    accept = (accept or "text/csv").split(";")[0].strip()
    values = [float(p) for p in np.atleast_1d(prediction)]
    if accept == "application/json":
        return json.dumps(values), "application/json"
    return "\n".join(str(v) for v in values), "text/csv"
"""

# log_model_artifacts preserves the local directory layout, so code/inference.py
# lands at <artifact_location>/code/inference.py.
with tempfile.TemporaryDirectory() as tmp:
    os.makedirs(os.path.join(tmp, "code"))
    with open(os.path.join(tmp, "code", "inference.py"), "w") as f:
        f.write(INFERENCE_SCRIPT)
    mlflow_client.log_model_artifacts(logged_model.model_id, tmp)

print(f"Logged code/inference.py to {logged_model.artifact_location}/code/")

#### Log the inference specification

`sagemaker_mlflow.log_inference_specification()` attaches the container image, the model data source, and the environment variables to the logged model. The instance types you list constrain where the Model Package can be deployed, so include the type you intend to use.

In [ ]:
inference_spec = {
    "Containers": [{
        "Image": xgboost_image,
        # Serve straight from the MLflow artifact store — no repacking, no copies.
        "ModelDataSource": {
            "S3DataSource": {
                "S3Uri": logged_model.artifact_location.rstrip("/") + "/",
                "S3DataType": "S3Prefix",
                "CompressionType": "None",
            }
        },
        # Point the BYOC serving stack at the script logged above.
        "Environment": {
            "SAGEMAKER_PROGRAM": "inference.py",
            "SAGEMAKER_SUBMIT_DIRECTORY": "/opt/ml/model/code",
        },
    }],
    "SupportedContentTypes": ["text/csv", "application/json"],
    "SupportedResponseMIMETypes": ["text/csv", "application/json"],
    "SupportedRealtimeInferenceInstanceTypes": ["ml.m5.large", "ml.m5.xlarge", "ml.r5.xlarge"],
    "SupportedTransformInstanceTypes": ["ml.m5.large", "ml.m5.xlarge"],
}

sagemaker_mlflow.log_inference_specification(
    logged_model.model_id, inference_specification=inference_spec
)
print("Inference specification logged on the MLflow model.")

In [ ]:
logged_model_artifacts = f"{presigned_url.split('/auth')[0]}/#/experiments/{experiment_id}/models/{logged_model.model_id}/artifacts?workspace=default"

In [ ]:
display(HTML(
    f'<b><a target="_blank" rel="noopener noreferrer" '
    f'href="{escape(logged_model_artifacts, quote=True)}">'
    'Open the registered model artifacts in MLflow</a></b>'
))

![Inference Specification for SageMaker logged as part of the model artifacts in MLflow](./img/inference-spec-mlflow.png)

#### Register, and watch the auto-sync

Register the logged model. This is the only registration in the notebook, so it becomes version 1 of `registered_model_name`. Two details make the artifact review-ready from the start:

- **Group tags.** `mlflow.register_model()` creates the registered model without tags. Creating it first with `MlflowClient.create_registered_model(tags=...)` lets the sync create the SageMaker **Model Package Group** with group-level facts (project, use case, owning team).
- **The sync tag.** The sync to the SageMaker Model Registry runs asynchronously and records the resulting Model Package ARN as a `sagemaker.model_package_arn` tag on the MLflow model version, a simpler way to find the package than searching the registry.

In [ ]:
from mlflow.exceptions import MlflowException

# Facts about the model as a whole: they become tags of the Model Package Group.
group_tags = {
    "project": "from-idea-to-prod",
    "use-case": "bank-marketing",
    "team": "data-science",
    "model-type": "xgboost-binary-classifier",
}
try:
    mlflow_client.create_registered_model(
        registered_model_name,
        tags=group_tags,
        description="Bank-marketing propensity model (XGBoost), registered from notebook 01.",
    )
    print(f"Created registered model '{registered_model_name}' with tags {sorted(group_tags)}")
except MlflowException as e:
    if e.error_code != "RESOURCE_ALREADY_EXISTS":
        raise
    print(f"Registered model '{registered_model_name}' already exists — reusing it")

registered_model_version = mlflow.register_model(
    f"models:/{logged_model.model_id}", registered_model_name
)
print(f"Registered in MLflow: {registered_model_version.name} v{registered_model_version.version}")

model_package_arn = None
for _ in range(20):
    mv = mlflow_client.get_model_version(
        registered_model_version.name, registered_model_version.version
    )
    model_package_arn = mv.tags.get("sagemaker.model_package_arn")
    if model_package_arn:
        break
    print(".", end="", flush=True)
    time.sleep(3)
if not model_package_arn:
    raise TimeoutError("SageMaker Model Package was not auto-created within the timeout")
print(f"\nSynced SageMaker Model Package: {model_package_arn}")

#### Inspect the auto-created Model Package

This is the object notebooks 03 and 04 build on. It already carries an `InferenceSpecification`, the payoff of logging the spec before registering.

<div class="alert alert-warning">
<b>SDK gap worth knowing about:</b> the typed <code>ModelPackage</code> resource deserializes the nested containers into a <code>ContainersItem</code> shape that models only 6 of the API's 16 container fields. <code>Environment</code>, <code>ImageDigest</code>, <code>ContainerHostname</code>, <code>Framework</code> and others are dropped on read, and touching <code>container.environment</code> raises <code>AttributeError</code> even though the values <i>are</i> stored in the registry. So the cell below reads the modeled fields from the typed resource and falls back to the raw <code>DescribeModelPackage</code> response for the rest.
</div>

In [ ]:
model_package = ModelPackage.get(model_package_name=model_package_arn)

# DescribeModelPackage does not return ModelPackageName for versioned packages,
# which breaks refresh()/update(). The API accepts an ARN in that field, so
# backfill it here.
model_package.model_package_name = model_package.model_package_arn

# The typed ContainersItem shape omits Environment, ImageDigest and friends, so
# read the raw response for those. One extra API call, no guessing.
described = boto3.client("sagemaker").describe_model_package(
    ModelPackageName=model_package_arn
)
raw_container = described["InferenceSpecification"]["Containers"][0]

print(f"Model package group:  {model_package.model_package_group_name}")
print(f"Version:              {model_package.model_package_version}")
print(f"Status:               {model_package.model_package_status}")
print(f"Approval status:      {described.get('ModelApprovalStatus', '(not set)')}")
print(f"Has inference spec:   {model_package.inference_specification is not None}")

print("\nInference specification, as stored in the registry")
print(f"  image:              {raw_container['Image']}")
print(f"  image digest:       {raw_container.get('ImageDigest', '(none)')}")
print(f"  model data source:  {raw_container['ModelDataSource']['S3DataSource']['S3Uri']}")
print(f"  compression:        {raw_container['ModelDataSource']['S3DataSource']['CompressionType']}")
print(f"  environment:        {raw_container.get('Environment', {})}")
print(f"  content types:      {described['InferenceSpecification']['SupportedContentTypes']}")
print(f"  instance types:     {described['InferenceSpecification']['SupportedRealtimeInferenceInstanceTypes']}")

# Fail loudly if the environment we logged did not survive the sync: without
# SAGEMAKER_PROGRAM a framework container has no inference code to run.
expected_env = inference_spec["Containers"][0]["Environment"]
synced_env = raw_container.get("Environment", {})
mismatched = {k: (v, synced_env.get(k)) for k, v in expected_env.items() if synced_env.get(k) != v}
if mismatched:
    raise RuntimeError(
        f"Environment did not round-trip through the MLflow sync: {mismatched}. "
        "The endpoint would fall back to the container's defaults."
    )
print("\nEnvironment round-tripped through the sync intact.")

The sync also generated a **draft Model Card** on the package, pre-filled with the training metrics and hyperparameters of the winning run. It comes with the auto-registration; you did not ask for it.

In [ ]:
import json

card = json.loads(described["ModelCard"]["ModelCardContent"])
training = card["training_details"]["training_job_details"]

print(f"Model card status: {described['ModelCard']['ModelCardStatus']}")
print("\nTraining metrics recorded on the card:")
for m in training["training_metrics"]:
    print(f"  {m['name']:<22}{m['value']:.4f}")
print("\nHyperparameters recorded on the card:")
for h in training["hyper_parameters"]:
    if h["value"] != "None":
        print(f"  {h['name']:<22}{h['value']}")

#### Move the version to staging: the data scientist's step

A freshly synced version has no lifecycle stage and no approval status. The data scientist sets the stage without leaving MLflow: setting the alias **`sagemakerlifecycle-<stage>-<status>`** on the model version makes the sync update the Model Package's `ModelLifeCycle`. A version holds one lifecycle alias at a time, so `set_lifecycle()` removes any existing one first.

Where provenance goes is dictated by the API: SageMaker supports tag operations on Model Package **Groups** only (`AddTags` against a version ARN fails with `ValidationException`), so:

- **Group resource tags** carry model-wide facts (project, use case, team) — what IAM `aws:ResourceTag` conditions, Resource Groups, and cost allocation see.
- **`CustomerMetadataProperties` on the version** carry per-version facts — shown on the version in Studio, and read by the notebook 02 card automation as `EvaluationMetrics`.

In [ ]:
sm_client = boto3.client("sagemaker")

# Lifecycle stage, driven from MLflow by alias. The sync applies it to the
# Model Package with the MLflow app's service role, then we wait for it to land.
set_lifecycle(mlflow_client, registered_model_version.name, registered_model_version.version,
              stage="staging", status="pending")
lifecycle = wait_for_lifecycle(sm_client, model_package_arn, "staging", "pending")

# Provenance of THIS version -> CustomerMetadataProperties on the version.
source_run = mlflow_client.get_run(best_run_id)
version_metadata = set_version_metadata(sm_client, model_package_arn, {
    "RegistrationSource": "notebook-01",
    "RegisteredBy": user_profile_name or "unknown",
    "MlflowExperiment": experiment_name,
    "TrainingJob": source_run.data.tags.get("sagemaker.job_name", "n/a"),
    "ValidationAuc": f"{metric_of(logged_model, 'validation_auc'):.4f}",
})
print("\nRecorded on the version (CustomerMetadataProperties):")
for key in sorted(version_metadata):
    print(f"  {key:<24}{version_metadata[key][:70]}")

# Facts about the model as a whole -> resource tags on the Model Package GROUP.
group_resource_tags = tag_model_package_group(sm_client, model_package_arn, group_tags)
print("\nResource tags on the Model Package Group:")
for key in sorted(group_resource_tags):
    print(f"  {key:<24}{group_resource_tags[key]}")

print_lifecycle(sm_client, model_package_arn, "\nRegistry state after the data scientist is done:")

## Pass the approval gate

The Model Package exists, sits in `staging/pending`, and is not approved. That ordering is the point of a registry: you get a reviewable artifact first, and promotion and deployment are separate, deliberate steps taken by the right people.

The rest of the notebook is the gate: first as the *data scientist*, who tries to promote to production and is denied by IAM; then as the *governance officer*, who reviews the synced evidence, promotes to `production/active`, and approves it.

Deployment is deliberately not part of this notebook. [Notebook 04](04-inference-and-monitoring.ipynb) resolves the approved version from the registry, validates its logged `inference.py` in the kernel, deploys it to a real-time endpoint with data capture, and monitors it.

### Two attributes, two enforcement points

Lifecycle stage and approval status are separate governance attributes:

- **Lifecycle stage** (`ModelLifeCycle`: `staging` or `production`, status `pending` or `active`) says *where the model is in its life*.
- **Approval status** (`ModelApprovalStatus`: `PendingManualApproval`, `Approved`, `Rejected`) says *whether it may be deployed*.

Promotion to production and approval for deployment are the governance officer's calls, and IAM keeps the data scientist out of them. Because the two attributes are enforced at different points, the cells below check them differently:

| Attempt | Route | How it is checked here |
|---|---|---|
| Promote to `production/active` | MLflow alias → sync → `UpdateModelPackage` by the **MLflow app service role** | `iam:SimulateCustomPolicy` on the data-scientist policy: IAM returns `explicitDeny` |
| Set `ModelApprovalStatus=Approved` | `UpdateModelPackage` by the **calling role** | assume the data-scientist role and call: `AccessDeniedException` |

Simulation is IAM's own evaluation engine answering the same question a live call would, and it is how the [blog's reference notebook](https://github.com/aws-samples/genai-ml-platform-examples/blob/main/operations/sagemaker-mlflow-model-registry-goverance/01_single_account_governance.ipynb) demonstrates the lifecycle guardrail. Attach this policy to the data-scientist roles *and* to the MLflow app service role to close the alias route too.

#### Data scientist: denied

Both checks below **expect** a denial. If either comes back allowed, the guardrail is not in place and the cell raises so you notice.

In [ ]:
iam_client = boto3.client("iam")
ds_policy = load_persona_policy(
    DATA_SCIENTIST,
    region=region,
    account_id=boto3.client("sts").get_caller_identity()["Account"],
    model_package_group_prefix="from-idea-to-prod-model-",
)
print("Data-scientist guardrail:")
describe_guardrail({"Statement": deny_statements(ds_policy)})

In [ ]:
# 1. Lifecycle promotion to production — the alias route runs as the MLflow app's
#    service role, so ask IAM to decide the request instead of calling the API.
for stage, status in (("staging", "pending"), ("production", "active")):
    decision, source = simulate_lifecycle_decision(iam_client, ds_policy, stage, status)
    verdict = "DENIED" if decision == "explicitDeny" else "NOT DENIED"
    print(f"\n{DATA_SCIENTIST} -> lifecycle {stage}/{status}: {verdict} "
          f"(IAM decision '{decision}', evaluated by {source})")
    if stage == "production" and decision != "explicitDeny":
        raise RuntimeError(
            "The data-scientist policy did not deny promotion to production. Check the "
            "DenyProductionPromotion statement in governance/data_scientist_policy.json."
        )

In [ ]:
# 2. Approval status IS evaluated against the caller, so this one is denied live.
ds_sm = assume_persona(data_scientist_role_arn, DATA_SCIENTIST, region=region)
allowed, message = try_update_model_package(
    ds_sm, ModelPackageArn=model_package_arn, ModelApprovalStatus="Approved"
)
print(f"\n{DATA_SCIENTIST} -> ModelApprovalStatus=Approved: {'ALLOWED' if allowed else 'DENIED'}")
if allowed:
    raise RuntimeError(
        "The data-scientist role was allowed to approve the model. Check the "
        "DenyDeploymentApproval statement in governance/data_scientist_policy.json."
    )
print(f"  {message}")

print_lifecycle(sm_client, model_package_arn, "\nUnchanged, as it should be:")

#### Governance officer: review, promote, approve

The officer does not need MLflow open: the synced version already carries the training metrics, evaluation card, lineage, and the provenance the data scientist recorded. The cell prints what a reviewer looks at first, then makes two distinct decisions in order:

- `ModelLifeCycle = production/active`: the version is now a production-stage candidate in this group. Set through the lifecycle alias, same mechanism as staging.
- `ModelApprovalStatus = Approved`: the version may be deployed. This turns on the **Deploy** button in Studio and is what the notebook 02 card automation reacts to (card status `PendingReview` → `Approved`). Made as the governance-officer role, whose policy carries no deny.

The decision is also recorded on the version as `ApprovedBy` and `ApprovedAt`, so the audit trail sits next to the model, not only in CloudTrail.

In [ ]:
go_sm = assume_persona(governance_officer_role_arn, GOVERNANCE_OFFICER, region=region)

# --- Review: what the officer sees in the registry, without opening MLflow.
described = go_sm.describe_model_package(ModelPackageName=model_package_arn)
card = json.loads(described["ModelCard"]["ModelCardContent"])
training = card.get("training_details", {}).get("training_job_details", {})
card_metrics = {m["name"]: m["value"] for m in training.get("training_metrics", [])}
provenance = described.get("CustomerMetadataProperties", {})
print("Evidence on the Model Package version:")
print(f"  validation_auc:   {card_metrics.get('validation_auc', float('nan')):.4f}")
print(f"  card status:      {described['ModelCard']['ModelCardStatus']}")
print(f"  registered by:    {provenance.get('RegisteredBy')} from {provenance.get('RegistrationSource')}")
print(f"  training job:     {provenance.get('TrainingJob')}")
print(f"  inference spec:   {'present' if described.get('InferenceSpecification') else 'MISSING'}")
print(f"  group tags:       {model_package_group_tags(go_sm, model_package_arn)}")

# --- Promote: lifecycle to production/active, through the alias.
set_lifecycle(mlflow_client, registered_model_version.name, registered_model_version.version,
              stage="production", status="active")
wait_for_lifecycle(sm_client, model_package_arn, "production", "active")

In [ ]:
# --- Approve for deployment: a direct call, made as the governance officer.
summary = approve_for_deployment(
    go_sm,
    model_package_arn,
    approved_by=f"{GOVERNANCE_OFFICER}:{user_profile_name}",
    note=f"Validation AUC {card_metrics.get('validation_auc', 0):.4f} meets the 0.75 bar; inference spec and card reviewed.",
)
print_lifecycle(go_sm, model_package_arn, "\nAfter the governance officer's decision:")
assert summary["approval_status"] == "Approved" and summary["stage"] == "production"

In practice the governance officer usually does this in the **Studio Models UI**: open the version, check the *Train*, *Evaluate*, and *Lineage* tabs, then update the stage and approval status from the *Details* tab. Same API calls, same IAM evaluation.

In [ ]:
display(
    HTML('<b>See <a target="top" href="https://studio-{}.studio.{}.sagemaker.aws/registry/registered-models/{}/versions/version-{}/details">the promoted Model Package version</a> in the Studio UI</b>'.format(
            domain_id, region, model_package.model_package_group_name, model_package.model_package_version))
)

![Model Version Details and Custom Metadata](./img/model-version-details.png)

Back as yourself, confirm the registry reflects the officer's decision; the typed `ModelPackage` resource fetched earlier only needs a `refresh()`.

Optionally the officer can **freeze** the model: `freeze_model_package_group()` tags the group `frozen=true`, and [`governance/freeze_policy.json`](governance/freeze_policy.json) turns that tag into a deny on further `UpdateModelPackage` / `CreateModelPackage`. This workshop leaves the group unfrozen, because notebook 03 registers another version into it.

In [ ]:
model_package.refresh()
print(f"Approval status:  {model_package.model_approval_status}")
print(f"Lifecycle:        {model_package.model_life_cycle}")

# The data scientist still cannot undo the approval.
allowed, message = try_update_model_package(
    ds_sm, ModelPackageArn=model_package_arn, ModelApprovalStatus="Approved"
)
print(f"\n{DATA_SCIENTIST} -> re-approve: {'ALLOWED' if allowed else 'DENIED'}")

# To freeze the group instead of leaving it open, uncomment:
# freeze_model_package_group(go_sm, model_package_arn)

### Continue the lineage after deployment

The registry lineage is reviewable now, but no deployment or endpoint exists yet. [Notebook 04](04-inference-and-monitoring.ipynb) opens this version's lineage after the endpoint reaches `InService`, when the graph can connect the MLflow experiment and approved Model Package to the SageMaker model, deployment, and running endpoint.

## Summary

In this notebook you:

**Part 1 — explore in the notebook**
- Explored the dataset, engineered features, and compared local XGBoost experiments in managed MLflow
- Kept exploratory models out of the registry

**Part 2 — inspect, validate locally, then run remotely**
- Inspected the shared `preprocess()` source, called it locally, then ran the same function on managed compute with `@remote`
- Inspected `train()`, used one hyperparameter dictionary for local and managed calls, and selected only the managed result
- Inspected `explain()` and ran it only against the managed model so the governance evidence matches the version being registered
- Used S3 paths, MLflow run IDs, and the logged-model ID as explicit handoffs
- Added the inference code and specification before registering one deployable Model Package
- Confirmed the data-scientist denials, then promoted and approved the version as the governance officer

Notebook 03 wraps the same reusable functions with `@step`; notebook 04 deploys the approved package and demonstrates a native ProcessingJob for scheduled monitoring.

## Continue with notebook 02

Open [notebook 02](02-model-card-automation.ipynb). It picks up the Model Package you just registered and automates completion of its Model Card, for this version and every version the notebook 03 pipeline registers, keeping the card status in step with the approval workflow.

## Further development ideas for your real-world projects
- Learn more about [managing experiments with MLflow](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
- Try no-code [SageMaker Canvas](https://docs.aws.amazon.com/sagemaker/latest/dg/canvas.html) for analysis, automated ML, and predictions
- Improve training with [automated hyperparameter tuning](https://docs.aws.amazon.com/sagemaker/latest/dg/automatic-model-tuning.html)
- Attach the `sagemaker:ModelLifeCycle/stage` guardrail to the **MLflow app service role** to close the alias route, apply [`governance/freeze_policy.json`](governance/freeze_policy.json) to lock approved models, and give each persona its own SageMaker user profile (or account, per Part 2 of the [governance blog series](https://aws.amazon.com/blogs/machine-learning/govern-models-with-mlflow-and-amazon-sagemaker-ai-model-registry-sync-part-1/)) instead of assuming roles from one notebook

## Additional resources
- [Managing ML experiments with MLflow in SageMaker](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow.html)
- [MLflow Apps in SageMaker](https://docs.aws.amazon.com/sagemaker/latest/dg/mlflow-app-setup.html)
- [Using Docker containers with SageMaker](https://docs.aws.amazon.com/sagemaker/latest/dg/docker-containers.html)
- [SageMaker JupyterLab guide](https://docs.aws.amazon.com/sagemaker/latest/dg/studio-updated-jl.html)
- [Operationalize your Amazon SageMaker Studio notebooks as scheduled notebook jobs](https://aws.amazon.com/blogs/machine-learning/operationalize-your-amazon-sagemaker-studio-notebooks-as-scheduled-notebook-jobs/)
- [scikit-learn dataset transformations](https://scikit-learn.org/stable/data_transforms.html)
- [Amazon SageMaker Immersion Day](https://catalog.us-east-1.prod.workshops.aws/workshops/63069e26-921c-4ce1-9cc7-dd882ff62575/en-US)

### Hands-on examples
- [Official Amazon SageMaker Examples repository](https://github.com/aws/amazon-sagemaker-examples)
- [Community Amazon SageMaker Examples repository](https://github.com/aws/amazon-sagemaker-examples-community)

# Shutdown kernel

In [ ]:
%%html

<p><b>Shutting down your kernel for this notebook to release resources.</b></p>
<button class="sm-command-button" data-commandlinker-command="kernelmenu:shutdown" style="display:none;">Shutdown Kernel</button>
        
<script>
try {
    els = document.getElementsByClassName("sm-command-button");
    els[0].click();
}
catch(err) {
    // NoOp
}    
</script>